Optimisation de l'entrainement pour `focus` 
This is the same function as used in `10_Transfer_learning_what_networks.ipynb`
> ... TODO ... # TODO test without circular padding, with Adam, with no warmstart 

    model = torchvision.models.resnet18(weights=None)

In [1]:
from retinotopy import *
welcome()

Running on GPU :  Tesla V100-SXM2-32GB #GPU= 1
Running on Jean Zay with Tesla V100-SXM2-32GB with DATAROOT='/lustre/fsn1/projects/rech/fsx/uvb28bo/data' and USER='uvb28bo' 


------------------------------------------------------------------------------------
On date 2025-03-06, Running learning on host r6i2n8 with device cuda, pytorch==2.6.0
------------------------------------------------------------------------------------
Welcome on Linux-5.14.0-427.50.1.el9_4.x86_64-x86_64-with-glibc2.34


In [2]:
args = Params()
data_set_type = 'bbox' # Select your root between : 'boxes', 'focus', 'full'
print(f'{data_set_type=}')
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training

data_set_type='bbox'


# optimize meta-parameters

In [3]:
# print(path_save)
# %ls -l {path}*
# %rm {path} + '.sqlite3'

In [4]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [5]:
study_name = datetag + '_optuna'

#  TODO study_name = datetag + '_optuna-Adam'

model_name = 'resnet101'
do_polar = True

model_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '.pt'


scan_dicts= {'lr' : [0.00015], 'beta2' : [0, .1], 'momentum' : [0.06]   }
scan_dicts= {'beta2' : [0, .1]  }

label_dicts= { 'lr' : 'lr',  'beta2' : 'Adam beta2', 'momentum' : 'momentum'}

In [6]:
model_filename

'cached_data/2025-03-06_bbox_resnet101_retino.pt'

In [7]:
%ls -l {model_filename}

-rw-r----- 1 uvb28bo genevg01 178839508 Mar 16 21:01 cached_data/2025-03-06_bbox_resnet101_retino.pt


In [8]:
subplotpars_scan = SubplotParams(left=0.125, right=.95, bottom=0.25, top=.975)
max_threshold = .999
for key in scan_dicts:
    filename = f'{data_cache}/{study_name}_{key}.json'

        
    if os.path.isfile(filename):
        df_scan = pd.read_json(filename)
    else:
        print(50*'=')
        print('Scanning along', key, "=", label_dicts[key])
        print(50*'=')
        
        measure_columns = [key, 'accuracy']
        df_scan = pd.DataFrame([], columns=measure_columns)
        i_loc = 0
        for i_value, value in enumerate(scan_dicts[key]):
            print('i_value', i_value + 1, ' /', len(scan_dicts[key]), key, '=', value)

            opt =  Params()

            new_dict = asdict(opt)
            new_dict[key] = value
            new_opt = Params(**new_dict)
            new_opt.n_train_stop = 1e5
            new_opt.num_epochs = 1
            new_opt.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training
            
            def objective(trial):
                # TODO : add the other parameters
                # new_opt.rs_min = trial.suggest_float('rs_min', -1, 1.)
                # new_opt.rs_max = trial.suggest_float('rs_max', -7, -4)
                scale = 10
                new_opt.momentum = trial.suggest_float('momentum', opt.momentum/scale, min(opt.momentum*scale, max_threshold), log=True)
                if new_opt.beta2>0: new_opt.beta2 = trial.suggest_float('beta2', new_opt.beta2/scale, min(new_opt.beta2*scale, max_threshold), log=True)
                scale = 50
                new_opt.lr = trial.suggest_float('lr', opt.lr / scale, opt.lr * scale, log=True)
                # new_opt.lr = trial.suggest_float('im_mean', opt.im_mean / scale, opt.im_mean * scale, log=True)
                # new_opt.lr = trial.suggest_float('im_std', opt.im_std / scale, opt.im_std * scale, log=True)
                new_opt.batch_size = trial.suggest_int('batch_size', 16, 512, log=True, step=1)

                # get the architecture of the network
                model_retrain = load_model(model_name=model_name, model_path=model_filename, do_scratch=new_opt.do_scratch, do_circular=new_opt.do_polar, verbose=False).to(device)
                                
                # load the data
                dataloaders = datasets_transforms(new_opt, verbose=False)

                # train and get accuracy on the validation set
                _, df_train = train_model(new_opt, model_retrain, dataloaders=dataloaders, verbose=False)
                
                accuracy = df_train['avg_acc_val'].mean()
                
                return accuracy


            # 3. Create a study object and optimize the objective function.
            sampler = optuna.samplers.TPESampler(multivariate=True)
            opt_tuna= dict(storage=f"sqlite:///{os.path.join(data_cache, study_name)}.sqlite3", sampler=sampler, direction='maximize', load_if_exists=True, study_name=f"{key} = {value}")
            study = optuna.create_study(**opt_tuna)
            study.optimize(objective, n_trials=max((150-len(study.trials), 0)), n_jobs=1, show_progress_bar=True)
            print(50*'-.')
            print("Best params: ", study.best_params)
            print("Best value: ", study.best_value)
            print("Best Trial: ", study.best_trial)
            print("Trials: ", study.trials)
            print(50*'-.')
            df_scan.loc[i_loc] = {key:value, 'accuracy':study.best_value}
            i_loc += 1
        df_scan.to_json(filename, orient='index', indent=2)
    print(df_scan)
    print(50*'=')

    fig, ax = plt.subplots(figsize=(fig_width, fig_width/phi), subplotpars=subplotpars_scan)
    gp_scan = df_scan[[key, 'accuracy']].groupby([key])
    means = gp_scan.mean()
    errors = gp_scan.std()
    means.plot.bar(yerr=errors, ax=ax, capsize=4, rot=-60, legend=False, color='r', alpha=.5)
    
    ax.set_ylabel('Accuracy')
    ax.set_xlabel(key + ' = ' +label_dicts[key])
    #ax.set_xscale('log')

    ax.set_ylim(0, 1)
    #fig = ax.get_figure()
    # pos = ax.get_position()
    # print(pos)
    plt.show()

Scanning along beta2 = Adam beta2
i_value 1  / 2 beta2 = 0


/lustre/fshomisc/sup/hpe/pub/miniforge/24.9.0/envs/pytorch-gpu-2.6.0+py3.12.8/lib/python3.12/site-packages/optuna/_experimental.py:31: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(


  0%|          | 0/56 [00:00<?, ?it/s]

loading .... cached_data/2025-03-06_bbox_resnet101_retino.pt


[W 2025-04-26 11:39:20,802] Trial 94 failed with parameters: {'momentum': 0.13181399170560668, 'lr': 0.00023712221308547343, 'batch_size': 212} because of the following error: OutOfMemoryError('CUDA out of memory. Tried to allocate 42.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 26.69 MiB is free. Including non-PyTorch memory, this process has 31.70 GiB memory in use. Of the allocated memory 30.86 GiB is allocated by PyTorch, and 494.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)').
Traceback (most recent call last):
  File "/lustre/fshomisc/sup/hpe/pub/miniforge/24.9.0/envs/pytorch-gpu-2.6.0+py3.12.8/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                     

OutOfMemoryError: CUDA out of memory. Tried to allocate 42.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 26.69 MiB is free. Including non-PyTorch memory, this process has 31.70 GiB memory in use. Of the allocated memory 30.86 GiB is allocated by PyTorch, and 494.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)